# <span style="color:#F57C00">What-if: Cooling a District with Green Space</span>

**What this notebook does.** The final model of the modeling notebook predicts the summer surface temperature of the built-up area of a district. Here we use it as a planning tool:
1. **What-if:** the same district with a different green share, everything else unchanged: how much cooler (or hotter) does it get, and how sure are we?
2. **Check against reality:** does the model agree with what really happened in districts whose green share changed between 2013-2015 and 2023-2025?
3. **Heat-target green need:** how many points of green share, and how many hectares, does a district need for a target cooling?

**Read this first.** The model learns from the differences *between districts*, and the what-if moves one district along those differences. It tells what the data associate with more green; it is **not an experiment** and not a promise. The number is for a *typical recent summer* (the weather is the mean of the 5 summers before), and it is for the **built-up area** (the surface temperature of the buildings and streets, not the air temperature).

**The five parts**

| Part | What happens |
| :--- | :--- |
| **1. Setup** | Load the final model, rebuild the model table, build one base row per district |
| **2. What-if engine** | One small function `what_if` and the curves of all districts (-20 to +30 points of green share) |
| **3. Uncertainty** | A bootstrap of the whole model gives a range around every curve |
| **4. Check against real changes** | The model against what really changed between 2013-2015 and 2023-2025, with four-period, control and panel checks |
| **5. Heat-target green need** | Points and hectares of green for -1 °C and for "to the city median"; saves the results |

**Results** are written to `data/results/`: `what_if_curves.csv` (the curves for a slider) and a first part of `district_scorecard.csv` (the risk notebook completes it). The checks of Part 4 also write `data/models/observational_validation.json` and `historical_changes.csv`, and the model card gets the observed reference that the application shows next to the model number.

**Run time** about 3 minutes on 16 CPU cores (the bootstrap of Part 3 is the long part).

# <span style="color:#F57C00">Part 1: Setup</span>

We load the final model and rebuild **exactly the same model table** as in the modeling notebook (Part 1 is copied on purpose, so that this notebook can be read from top to bottom). A check in Step 3 stops the notebook if the features ever differ from the ones the model was trained with.

## Step 1: Libraries, settings and the tables

In [ ]:
import hashlib
import json
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

PROCESSED = Path("../data/processed")
MODELS = Path("../data/models")
RESULTS = Path("../data/results")
TARGET = "landsat_lst_c"  # the summer surface temperature of the built-up area (°C)
SEED = 2025  # the random draws of the bootstraps in Parts 3 and 4

green = pd.read_csv(PROCESSED / "greenness_districts_summer_2013_2025.csv")
climate = pd.read_csv(PROCESSED / "climate_districts_monthly_2000_2025.csv")
population = pd.read_csv(PROCESSED / "tuik_districts_population_clean.csv")
static = pd.read_csv(PROCESSED / "districts_urban_static.csv")
districts = gpd.read_file(PROCESSED / "turkey_districts_master.geojson")
centers = pd.DataFrame(
    districts[["district_code", "center_lat", "center_lon"]]
)  # only the center points, without the polygons
print(
    f"greenness {green.shape} | climate {climate.shape} | population {population.shape} | static {static.shape} | centers {centers.shape}"
)

## Step 2: The model table (a copy of Part 1 of the modeling notebook)
Weather and population features, the joins, the reliable rows, the regions and the within-city columns are built exactly as there:
- **Reliable rows:** a built-up area of at least 0.5 km² and at least half of its pixels clear; only cities with at least 3 such districts in a year stay.
- **Within-city columns** (`..._vs_city`): the value minus the mean of the same city in the same year. `add_within_city` is a small function because the what-if needs it again.
- **Two more feature groups** (Part 5 of the modeling notebook): the share of clear pixels of the summer (`landsat_cover`, with its within-city column) and the climate normals of 2000-2012 (`precip_clim`, `temp_clim`). Together with the 26 features of Part 4 they are the 30 features of the final model.

In [ ]:
summer = (
    climate[climate["month"].isin([7, 8])]
    .groupby(["district_code", "year"])
    .agg(temp_mean_ja=("temp_mean", "mean"), hot_days_ja=("hot_days", "sum"))
    .reset_index()
)
spring = (
    climate[climate["month"].between(3, 6)]
    .groupby(["district_code", "year"])
    .agg(precip_spring=("precip_sum", "sum"))
    .reset_index()
)
weather = summer.merge(
    spring, on=["district_code", "year"], validate="one_to_one"
)  # all districts and years

age_65_plus = [
    c
    for c in population.columns
    if c.startswith("total_")
    and c.split("_")[1][:2] in ("65", "70", "75", "80", "85", "90")
]
population["share65"] = population[age_65_plus].sum(axis=1) / population["total_pop"]
population["logpop"] = np.log(population["total_pop"])

table = (
    green.merge(
        weather, on=["district_code", "year"], how="left", validate="one_to_one"
    )
    .merge(
        population[["district_code", "year", "share65", "logpop"]],
        on=["district_code", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        static[["district_code", "elevation_m", "built_share_2020"]],
        on="district_code",
        how="left",
        validate="many_to_one",
    )
    .merge(centers, on="district_code", how="left", validate="many_to_one")
)
assert len(table) == len(green), "a join must not change the number of rows"

REGIONS = {
    "Marmara": [10, 11, 16, 17, 22, 34, 39, 41, 54, 59, 77],
    "Aegean": [3, 9, 20, 35, 43, 45, 48, 64],
    "Mediterranean": [1, 7, 15, 31, 32, 33, 46, 80],
    "Central Anatolia": [6, 18, 26, 38, 40, 42, 50, 51, 58, 66, 68, 70, 71],
    "Black Sea": [5, 8, 14, 19, 28, 29, 37, 52, 53, 55, 57, 60, 61, 67, 69, 74, 78, 81],
    "Eastern Anatolia": [4, 12, 13, 23, 24, 25, 30, 36, 44, 49, 62, 65, 75, 76],
    "Southeastern Anatolia": [2, 21, 27, 47, 56, 63, 72, 73, 79],
}
region_of_city = {city: region for region, cities in REGIONS.items() for city in cities}

WEATHER = ["temp_mean_ja", "hot_days_ja", "precip_spring"]
WITHIN_CITY = [
    "landsat_green_share",
    "elevation_m",
    "built_share_2020",
    "log_urban_km2",
    "logpop",
    "share65",
    "temp_mean_ja",
    "hot_days_ja",
]


def add_within_city(frame, columns):
    # every value minus the mean of the same city in the same year
    frame = frame.copy()
    city_means = frame.groupby(["city_code", "year"])[columns].transform("mean")
    frame[[f"{name}_vs_city" for name in columns]] = (
        frame[columns] - city_means
    ).to_numpy()
    return frame


table["log_urban_km2"] = np.log(table["urban_km2"])
reliable = (
    (table["urban_km2"] >= 0.5)
    & (table["landsat_cover"] >= 0.5)
    & table[TARGET].notna()
)
table = table[reliable].dropna(subset=["share65", "logpop"])
table = table[
    table.groupby(["city_code", "year"])["district_code"].transform("size") >= 3
]

region_columns = pd.get_dummies(
    table["city_code"].map(region_of_city), prefix="region", dtype=int
)[[f"region_{name}" for name in REGIONS]]
table = add_within_city(
    pd.concat([table, region_columns], axis=1), WITHIN_CITY
).reset_index(drop=True)

FEATURES = (
    [
        "landsat_green_share",
        "elevation_m",
        "built_share_2020",
        "log_urban_km2",
        "logpop",
        "share65",
        "temp_mean_ja",
        "hot_days_ja",
        "precip_spring",
        "center_lat",
        "center_lon",
    ]
    + list(region_columns.columns)
    + [f"{name}_vs_city" for name in WITHIN_CITY]
)

# the two feature groups of Part 5 of the modeling notebook: clear sky and the climate normals of 2000-2012 (before the first summer of the table)
WITHIN_CITY_NEW = WITHIN_CITY + [
    "landsat_cover"
]  # the within-city columns of the final model also have the clear sky
early = climate[climate["year"] <= 2012]
normals = pd.DataFrame(
    {
        "precip_clim": early.groupby(["district_code", "year"])["precip_sum"]
        .sum()
        .groupby("district_code")
        .mean(),
        "temp_clim": early.groupby("district_code")["temp_mean"].mean(),
    }
).reset_index()
table = add_within_city(
    table.merge(normals, on="district_code", how="left", validate="many_to_one"),
    WITHIN_CITY_NEW,
)
ADDED_FEATURES = ["landsat_cover", "landsat_cover_vs_city", "precip_clim", "temp_clim"]
FEATURES = FEATURES + ADDED_FEATURES
MAX_GREEN = table[
    "landsat_green_share"
].max()  # the highest green share in the data: a tree model cannot predict beyond what it has seen
print(
    f"{len(table)} rows | {table['district_code'].nunique()} districts | {table['city_code'].nunique()} cities | {len(FEATURES)} features"
)

## Step 3: The final model and its model card
The final model is **one XGBoost model** that learns the surface temperature minus half of the air temperature of the summer and adds that half back when it predicts (Part 5 of the modeling notebook). The small class and the settings are copied here on purpose, because the bootstrap of Part 3 trains the whole model again and a check in Part 4 trains it on the early years only. The model card was saved with the model; if the features of this notebook differ from the ones in the card, the notebook must stop: the model would be fed the wrong columns.

In [ ]:
GREEN = (
    "landsat_green_share",
    "landsat_green_share_vs_city",
)  # more green must never mean a hotter surface


def make_xgb(features, params, seed=0, n_jobs=1):
    # XGBoost with the settings of Part 4; the green share may only lower the prediction
    monotone = tuple(-1 if name in GREEN else 0 for name in features)
    return XGBRegressor(
        **params,
        monotone_constraints=monotone,
        tree_method="hist",
        n_jobs=n_jobs,
        random_state=seed
    )


class AirTemperatureOffset(RegressorMixin, BaseEstimator):
    """The final model. `estimator` learns the surface temperature minus a share of the air temperature of the summer;
    the share is added back when predicting. A hotter summer then moves the prediction directly, which trees cannot do
    on their own beyond the summers they have seen. `RegressorMixin` must come first: scikit-learn only recognizes a
    class as a regressor in that order, and cross-validation helpers check it. The state is explicit both ways:
    `fit` sets `estimator_`, and `from_loaded` sets it for a model that was read from a file."""

    def __init__(self, estimator, share=0.5, column="temp_mean_ja"):
        self.estimator, self.share, self.column = estimator, share, column

    def fit(self, X, y):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        self.estimator_ = clone(self.estimator).fit(X, y - self.share * X[self.column])
        return self

    def predict(self, X):
        estimator = getattr(self, "estimator_", None)  # a model loaded from a file has no fitted state yet
        if estimator is None:
            raise ValueError("the model is neither fitted nor loaded")
        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError("the prediction columns differ from the fitted model")
        return estimator.predict(X) + self.share * X[self.column].to_numpy()

    @classmethod
    def from_loaded(cls, estimator, features, share=0.5, column="temp_mean_ja"):
        """A native XGBoost file read from disk, with its fitted state made explicit."""
        booster_names = estimator.get_booster().feature_names
        if booster_names is not None and list(booster_names) != list(features):
            raise ValueError("the model file uses other feature names than the model card")
        model = cls(estimator, share, column)
        model.estimator_ = estimator
        model.feature_names_in_ = np.asarray(features, dtype=object)
        model.n_features_in_ = len(features)
        return model


def new_final(features, params, seed=0, n_jobs=1):
    return AirTemperatureOffset(make_xgb(features, params, seed, n_jobs))


def load_final(card, folder):
    # the final model from the files of the model card, without training
    trees = XGBRegressor()
    trees.load_model(folder / card["model_file"])  # the native format of XGBoost
    return AirTemperatureOffset.from_loaded(
        trees, card["features"], card["air_temperature_share"],
        card["air_temperature_column"],
    )

In [ ]:
card = json.loads((MODELS / "model_card.json").read_text())
assert (
    "air_temperature_share" in card
), "this notebook is written for the final model of the modeling notebook"
assert (
    FEATURES == card["features"]
), "the features of this notebook differ from the model card"
assert (
    len(table) == card["training_rows"]
), "the model table differs from the one the model was trained on"
model = load_final(card, MODELS)  # the model file in the native format of XGBoost

predicted = model.predict(table[FEATURES])
print(
    f"{card['algorithm']}, {len(FEATURES)} features, trained on {card['trained_on_years'][0]}-{card['trained_on_years'][1]}"
)
print(
    f"exam 2023-2025 (modeling notebook): RMSE {card['exam_2023_2025_real_weather']['rmse']:.2f} °C, R² inside the cities {card['exam_2023_2025_real_weather']['r2_within']:.2f}"
)
print(
    f"mean measured {table[TARGET].mean():.2f} °C | mean predicted {predicted.mean():.2f} °C (in-sample, the model has seen these rows)"
)

## Step 4: One base row per district
The what-if needs a starting point for every district:
- **The row:** its **latest reliable summer** (2025 for most districts), so the green share, the buildings and the population are the most recent ones.
- **The weather:** the **scenario weather**, the mean of the 5 summers before that year (the same rule as the weather scenario of the modeling notebook). The result is then for a *typical recent summer* and not for the luck of one summer.
- The within-city columns depend on the weather, so they are calculated again for the scenario. This is done for **all rows** of the table, so that every city in every year stays complete; `n_districts` (how many districts the city has in that year) is kept because the what-if needs it.

In [ ]:
weather = weather.sort_values(["district_code", "year"])
scenario_weather = weather[["district_code", "year"]].copy()
for name in WEATHER:
    scenario_weather[name] = weather.groupby("district_code")[name].transform(
        lambda s: s.shift(1).rolling(5).mean()
    )  # shift(1): without the year itself

scenario_table = table.drop(columns=WEATHER).merge(
    scenario_weather, on=["district_code", "year"], how="left", validate="many_to_one"
)
scenario_table = add_within_city(scenario_table, WITHIN_CITY_NEW)
scenario_table["n_districts"] = scenario_table.groupby(["city_code", "year"])[
    "district_code"
].transform("size")
scenario_table["region"] = scenario_table["city_code"].map(region_of_city)

is_latest = scenario_table["year"] == scenario_table.groupby("district_code")[
    "year"
].transform("max")
base = scenario_table[is_latest].reset_index(drop=True)
base["base_lst"] = model.predict(base[FEATURES])

assert (
    len(scenario_table) == len(table)
    and (
        scenario_table["district_code"].to_numpy() == table["district_code"].to_numpy()
    ).all()
), "the join must keep the rows and their order"
assert (scenario_table["n_districts"] >= 3).all()
assert len(base) == base["district_code"].nunique() == 829, "one base row per district"
assert (
    base[FEATURES].notna().all().all()
), "the scenario weather must exist for every base row"
assert base.set_index("district_code")["year"].equals(
    table.groupby("district_code")["year"].max()
), "the base row must be the latest year"

print(f"{len(base)} base rows, {base['city_code'].nunique()} cities")
print(
    "latest year of the districts:", base["year"].value_counts().sort_index().to_dict()
)
real = table.merge(
    base[["district_code", "year"]]
)  # the same districts and years with their real weather
print(
    f"mean temperature of the summer: scenario {base['temp_mean_ja'].mean():.1f} °C against real {real['temp_mean_ja'].mean():.1f} °C | hot days: scenario {base['hot_days_ja'].mean():.0f} against real {real['hot_days_ja'].mean():.0f}"
)
print(
    f"mean surface temperature: predicted with the scenario weather {base['base_lst'].mean():.2f} °C | measured with the real weather {real[TARGET].mean():.2f} °C"
)

**What we learned (Part 1)**
- The table is the same as in the modeling notebook (the check against the model card passed), with 829 districts in 80 cities. The other 144 districts get no what-if: 142 have a measured built-up area under 0.5 km² in every year (small towns, mostly on the Black Sea and in the east; together only 1.3% of the urban population), and the 2 reliable districts of Bayburt are left out because a city needs at least 3 reliable districts for the comparison inside the city (so Bayburt is the one province without a district).
- Each district has one base row: its latest reliable summer (2025 for 827 districts, 2024 for 2) with the scenario weather. From here on every what-if starts from these 829 rows.
- **The scenario is a typical summer, not 2025:** 2025 was hotter than the five summers before (25.2 °C and 38 hot days against 24.4 °C and 32), so the scenario prediction (42.74 °C) is about 0.7 °C below the measured 2025 mean (43.45 °C). What-if *changes* are small compared with this weather effect, which is why we always show them as a change (`effect`) and not as a forecast of one summer.

# <span style="color:#F57C00">Part 2: The What-if Engine</span>

## Step 1: How one district is changed
We raise (or lower) the green share of **one district**, keep everything else as it is, and let the model predict again. The model has two columns that depend on the green share, so both must change:
- `landsat_green_share` changes by the full change `delta`;
- `landsat_green_share_vs_city` (the share minus the city mean) changes by `delta * (1 - 1/n)`: the district moves by `delta`, but the mean of its city moves by `delta / n`, with `n` the number of districts of the city.

Two limits protect the answer, because a tree model **cannot predict outside the data it has seen**: a share is never above the highest share in the data (`MAX_GREEN`, about 0.92) and never below 0. The change that is really applied (`applied`) can therefore be smaller than the requested one, and the within-city column uses the applied change. Later we also limit the change to +30 points.

`what_if` is the function that the whole notebook uses. (The only other small functions are `add_within_city` from the modeling notebook and, in Part 5, `points_needed`.)

In [ ]:
def what_if(rows, deltas, model=model):
    # rows: base rows (any subset); deltas: changes of the green share (0.10 = +10 points)
    # returns four arrays (one row per district, one column per delta): the new green share, the predicted surface temperature, the effect (the change against the unchanged row) and the change that was really applied (equal to the requested one except at the limits)
    deltas = np.asarray(deltas)
    share = rows["landsat_green_share"].to_numpy()[:, None]
    new_share = np.clip(share + deltas, 0, MAX_GREEN)
    applied = new_share - share
    n = rows["n_districts"].to_numpy()[:, None]

    stacked = pd.concat(
        [rows[FEATURES]] * len(deltas), ignore_index=True
    )  # one block of rows per delta, so the model predicts everything in one call
    stacked["landsat_green_share"] = new_share.T.ravel()
    stacked["landsat_green_share_vs_city"] += (applied * (1 - 1 / n)).T.ravel()
    predicted = model.predict(stacked).reshape(len(deltas), len(rows)).T
    return (
        new_share,
        predicted,
        predicted - model.predict(rows[FEATURES])[:, None],
        applied,
    )


three = base.head(3)
new_share, predicted, effect, applied = what_if(three, [0.10])
pd.DataFrame(
    {
        "district": three["district_name"],
        "green_share": three["landsat_green_share"],
        "new_share": new_share[:, 0],
        "predicted_lst": predicted[:, 0],
        "effect": effect[:, 0],
    }
)

## Step 2: Is the shortcut exact?
The `(1 - 1/n)` shortcut is only worth using if it gives the same answer as the honest way: change the district in the **whole table**, calculate all within-city columns again with `add_within_city`, and predict. We compare both for 30 districts, including all 6 districts whose city has only 3 districts (where the city mean moves most) and the districts at the limits.

In [ ]:
rng = np.random.default_rng(0)
small_cities = base[base["n_districts"] == 3].index.to_numpy()
at_limit = base[(base["landsat_green_share"] + 0.10 > MAX_GREEN)].index.to_numpy()
others = rng.choice(base.index.to_numpy(), size=20, replace=False)
checked = np.unique(np.concatenate([small_cities, at_limit, others]))

fast = what_if(base.loc[checked], [0.10])[2][:, 0]
slow = []
for i in checked:
    row = base.loc[i]
    copy = scenario_table.copy()
    mine = (copy["district_code"] == row["district_code"]) & (
        copy["year"] == row["year"]
    )
    copy.loc[mine, "landsat_green_share"] = min(
        row["landsat_green_share"] + 0.10, MAX_GREEN
    )
    copy = add_within_city(
        copy, WITHIN_CITY_NEW
    )  # everything about the city is calculated again
    slow.append(
        model.predict(copy.loc[mine, FEATURES])[0]
        - model.predict(base.loc[[i], FEATURES])[0]
    )

assert np.allclose(
    fast, slow, atol=1e-9
), "the shortcut differs from the honest recalculation"
print(
    f"{len(checked)} districts checked ({len(small_cities)} with 3 districts in the city, {len(at_limit)} at the limit): largest difference {np.abs(fast - np.array(slow)).max():.2e} °C"
)

## Step 3: The curves of all districts
For every district we calculate the change of the surface temperature for green share changes of **-20 to +30 points in steps of 1 point** (51 steps). The steps of 5 points are saved for a slider; the fine steps are needed for the heat target in Part 5. `effect_grid` has one row per district and one column per step.

In [ ]:
GRID = np.arange(-20, 31)  # change of the green share in points
new_share, predicted, effect_grid, applied_grid = what_if(
    base, GRID / 100
)  # 829 districts x 51 steps; applied_grid is the change really applied (the same except at the limits)

steps = np.flatnonzero(GRID % 5 == 0)  # the steps of 5 points go to the slider table
curves = pd.DataFrame(
    {
        "district_code": np.repeat(base["district_code"].to_numpy(), len(steps)),
        "delta_pts": np.tile(GRID[steps], len(base)),
        "new_green_share": new_share[:, steps].ravel(),
        "predicted_lst": predicted[:, steps].ravel(),
        "effect": effect_grid[:, steps].ravel(),
    }
)
curves = base[["district_code", "district_name", "city_name"]].merge(
    curves, on="district_code", validate="one_to_many"
)

print(effect_grid.shape, "|", len(curves), "rows of the slider table")
print(
    {
        f"{d:+d} pts": round(float(effect_grid[:, list(GRID).index(d)].mean()), 2)
        for d in (-20, -10, 5, 10, 20, 30)
    },
    "(mean effect in °C)",
)

## Step 4: Does the engine behave?
Six tests that must hold, otherwise the notebook stops:
1. A change of 0 points gives an effect of exactly 0.
2. More green never means hotter: every effect of a positive change is $\le 0$, of a negative change $\ge 0$, and every curve is monotone (the monotone constraint of the model).
3. The new share stays between 0 and `MAX_GREEN`; where the limit is reached, the applied change is smaller than the requested one.
4. The mean effect of +10 points is the value of the model card (modeling notebook: the mean of three test years), within 0.1 °C.
5. The regions keep the main result of the EDA and the modeling notebook: the Southeast cools the most.
6. The curve flattens: the second +10 points cool less than the first.

In [ ]:
zero = list(GRID).index(0)
plus10 = list(GRID).index(10)
assert (effect_grid[:, zero] == 0).all(), "0 points must change nothing"
assert (effect_grid[:, zero + 1 :] <= 1e-12).all() and (
    effect_grid[:, :zero] >= -1e-12
).all(), "more green must never be hotter"
assert (np.diff(effect_grid, axis=1) <= 1e-12).all(), "every curve must be monotone"
assert ((new_share >= 0) & (new_share <= MAX_GREEN)).all()

capped = new_share[:, list(GRID).index(30)] == MAX_GREEN
assert (
    new_share[capped, list(GRID).index(30)] - base.loc[capped, "landsat_green_share"]
    < 0.30
).all(), "where the limit is reached, less than +30 points are applied"
print(
    f"+30 points: the limit {MAX_GREEN:.2f} is reached by {capped.sum()} districts ({capped.mean():.1%}); +10 points: {(new_share[:, plus10] == MAX_GREEN).sum()} districts"
)

mean10 = effect_grid[:, plus10].mean()
assert (
    abs(mean10 - card["what_if_plus_10_points_green_mean_change_c"]) < 0.1
), "different from the model card"
by_region = (
    pd.Series(effect_grid[:, plus10], index=base["region"])
    .groupby(level=0)
    .mean()
    .sort_values()
)
assert (
    by_region.index[0] == "Southeastern Anatolia"
), "the Southeast should cool the most (EDA and modeling notebook)"
second_10 = effect_grid[:, list(GRID).index(20)] - effect_grid[:, plus10]
assert (
    second_10.mean() > effect_grid[:, plus10].mean()
), "the second 10 points should cool less than the first"

print(
    f"+10 points: mean {mean10:+.2f} °C (model card {card['what_if_plus_10_points_green_mean_change_c']:+.2f}) | districts that get cooler: {(effect_grid[:, plus10] < 0).mean():.1%}"
)
third_10 = effect_grid[:, list(GRID).index(30)] - effect_grid[:, list(GRID).index(20)]
print(
    f"each further +10 points: first {mean10:+.2f}, second {second_10.mean():+.2f}, third {third_10.mean():+.2f} °C | losing 10 points: {effect_grid[:, list(GRID).index(-10)].mean():+.2f} °C"
)
print("by region:", by_region.round(2).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
low, middle, high = np.percentile(effect_grid, [10, 50, 90], axis=0)
axes[0].fill_between(
    GRID, low, high, color="#F57C00", alpha=0.25, label="10-90% of the districts"
)
axes[0].plot(
    GRID, effect_grid.mean(axis=0), color="#F57C00", lw=2, label="mean of the districts"
)
axes[0].plot(GRID, middle, color="black", lw=1, ls="--", label="median")
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].axvline(0, color="grey", lw=0.8)
axes[0].set_xlabel("change of the green share (points)")
axes[0].set_ylabel("change of the surface temperature (°C)")
axes[0].set_title("What-if curve of the districts")
axes[0].legend()

by_region.iloc[::-1].plot.barh(
    ax=axes[1], color="#F57C00"
)  # the strongest effect on top
axes[1].set_xlabel("mean change for +10 points of green (°C)")
axes[1].set_ylabel("")
axes[1].set_title("By region")
plt.tight_layout()
plt.show()

## Step 5: One district or the whole city?
The what-if changes **one district** and leaves the rest of its city as it is, so the district gets greener *relative to its neighbours*, and the model has a column for exactly that (`landsat_green_share_vs_city`). A planner may also ask: *what if all districts of a city get greener together?* Then the city mean moves with them and the within-city column does not change; only the raw green share does. We calculate this too, for +10 points, to see how much the answer depends on it. The calculation uses the **complete roster** of every city and year (all its districts that are in the table): the two districts whose latest reliable summer is 2024 must not be compared against a city of one.

In [ ]:
# The whole-city scenario uses the COMPLETE table: every district of every city-year gets greener
# together and the within-city columns are calculated again from the complete roster. For the two
# districts whose latest reliable summer is 2024 this matters: they must not be compared against a
# city of one. (The published version grouped only the latest base rows and met exactly that flaw.)
context = scenario_table.copy()
context["landsat_green_share"] = np.clip(
    context["landsat_green_share"] + 0.10, 0, MAX_GREEN
)
context["landsat_green_share_vs_city"] = context["landsat_green_share"] - context.groupby(
    ["city_code", "year"]
)["landsat_green_share"].transform(
    "mean"
)  # the city mean moves with the districts
selected = context.set_index(["district_code", "year"]).loc[
    base.set_index(["district_code", "year"]).index
]
whole_city = model.predict(selected[FEATURES]) - model.predict(base[FEATURES])

hotter = (
    whole_city > 1e-9
)  # districts that cannot follow the city (their share is at the limit of the data) fall behind it
assert (
    whole_city.mean() > effect_grid[:, plus10].mean()
), "greening a whole city should cool less than greening one district"
assert (
    hotter.sum() <= 10
    and (base.loc[hotter, "landsat_green_share"] > MAX_GREEN - 0.10).all()
    and (
        selected.loc[hotter, "landsat_green_share_vs_city"].to_numpy()
        < base.loc[hotter, "landsat_green_share_vs_city"].to_numpy()
    ).all()
), "only districts that cannot follow the city may get hotter"
print(
    f"+10 points: one district at a time {effect_grid[:, plus10].mean():+.2f} °C | all districts of the city together {whole_city.mean():+.2f} °C ({whole_city.mean() / effect_grid[:, plus10].mean():.0%} of it)"
)
print(
    f"districts that get hotter when the whole city gets greener: {hotter.sum()} (green share {base.loc[hotter, 'landsat_green_share'].round(2).tolist()}: they cannot follow +10 points and fall behind their city)"
)

**What we learned (Part 2)**
- The shortcut `delta * (1 - 1/n)` gives the same answer as recalculating the whole table (difference below $10^{-9}$ °C), also for cities with only 3 districts.
- **+10 points of green share lowers the surface temperature by 0.63 °C on average**, in every district (the monotone constraint). The modeling notebook found 0.63 °C with the models of the three test years and the real weather; this is the final model with the scenario weather and the latest districts. The effect is strongest in the Southeast (-0.71 °C) and the Black Sea (-0.70 °C) and weakest in Marmara (-0.47 °C); the middle 80% of the districts lie between -1.0 and -0.35 °C.
- **The curve flattens slowly:** the first +10 points cool by 0.63 °C, the second by 0.59 °C, the third by 0.56 °C; losing 10 points heats by 0.56 °C. A tree model gives a staircase, not a smooth line.
- **One district against the whole city:** when all districts of a city get greener together, the model gives about -0.45 °C per +10 points instead of -0.63 °C (71%). The within-city part of the effect only exists when a district becomes greener than its neighbours. Everything in this notebook is for **one district at a time**; a city-wide programme would cool about 30% less per point. The only districts that get slightly hotter in this test are 4 districts with a green share of 87-88%, close to the limit of the data: they cannot follow +10 points, fall behind their city, and the model reads that as relatively less green.
- A tree model can only say what it has seen: beyond the highest share in the data (36 districts reach the limit at +30 points) or beyond +30 points it would only repeat the last value, so we do not go there.
- These curves are the **central estimate**. Part 3 adds the uncertainty.

# <span style="color:#F57C00">Part 3: How Sure Are We? The Uncertainty of the Curves</span>

The curves of Part 2 come from **one** model, trained on one set of cities. A different sample of cities would give a slightly different model and a slightly different curve. A **bootstrap** shows how much:

1. Draw the cities **with replacement** from the 80 cities of the model table. A sampled city carries **all its districts and all its years together**, because districts of one city are neighbours that share climate and weather, and the years of one district are not independent of each other. This is also how the modeling notebook tests the model (whole cities always stay together there); a random split of rows would be far too optimistic (EDA).
2. Train the model again with the final settings on the drawn districts.
3. Calculate the curves of all districts with this model.
4. Repeat 60 times. For every district and every step, the **5th and 95th percentile** of the 60 curves give `effect_low` and `effect_high`. The 60 curves of every district are saved in `data/models/bootstrap_effects.npz`; the app takes its range from them (the central estimate it calculates live with the final model).

The range tells how much the answer depends on *which cities the model has seen*. It does not cover everything: the weather of the summer, errors in the green share, and the fact that the model finds an association and not a cause are not in it.

## Step 1: The bootstrap

In [ ]:
def base_fingerprint(rows, features):
    # a stable hash of the baseline rows, so the app can refuse a bank that belongs to other data
    columns = ["district_code", "year", "n_districts"] + list(features)
    frame = rows.reset_index(drop=True).sort_values("district_code")[columns]
    return hashlib.sha256(pd.util.hash_pandas_object(frame, index=False).values.tobytes()).hexdigest()


N_BOOT = 60
cities = np.sort(table["city_code"].unique())
by_city = table.set_index("city_code")

effects_boot = np.empty((N_BOOT, len(base), len(GRID)))  # draw x district x step
for b in range(N_BOOT):
    drawn = np.random.default_rng(SEED + b).choice(cities, size=len(cities), replace=True)
    sample = by_city.loc[
        drawn
    ].reset_index()  # a city that is drawn twice appears twice, with all its districts and years
    boot_model = new_final(FEATURES, card["settings"], seed=b, n_jobs=-1).fit(
        sample[FEATURES], sample[TARGET]
    )
    effects_boot[b] = what_if(base, GRID / 100, model=boot_model)[2]

# The app may reuse this bank only for exactly this model file and this baseline:
model_sha = hashlib.sha256((MODELS / card["model_file"]).read_bytes()).hexdigest()
base_hash = base_fingerprint(base, FEATURES)
np.savez_compressed(
    MODELS / "bootstrap_effects.npz",
    effects=effects_boot.astype(np.float32),
    district_code=base["district_code"].to_numpy(),
    grid=GRID,
    bootstrap_unit="city",
    seed=SEED,
    model_sha256=model_sha,
    base_fingerprint=base_hash,
    max_green=MAX_GREEN,
)  # the app reads the range of the curves from here (and the risk notebook the maximum share)
card["bootstrap"] = dict(
    unit="city", draws=int(N_BOOT), seed=int(SEED), model_sha256=model_sha, base_fingerprint=base_hash
)  # the record of the bank in the model card (the app reads the file itself)
(MODELS / "model_card.json").write_text(json.dumps(card, indent=1, default=float))
print(
    f"{N_BOOT} models trained, each on {len(sample)} rows ({sample['district_code'].nunique()} different districts in the last draw)"
)
print(f"bootstrap bank: {N_BOOT} city draws, model sha256 {model_sha[:12]}..., baseline {base_hash[:12]}...")

## Step 2: The range of every curve
The 5th and 95th percentile over the 60 draws. The central curve comes from the model that has seen **all** districts, and a bootstrap sample has seen only about two thirds of the cities, so the central curve can lie slightly outside the range. When it does, the range is **widened to include it** (a range that does not contain its own estimate would confuse anybody), and we count how often this happens.

In [ ]:
low, high = np.percentile(effects_boot, [5, 95], axis=0)
too_high = (low > effect_grid + 1e-9).mean()  # the central curve is below the range
too_low = (high < effect_grid - 1e-9).mean()  # the central curve is above the range
effect_low = np.minimum(low, effect_grid)
effect_high = np.maximum(high, effect_grid)

assert effects_boot.shape == (N_BOOT, len(base), len(GRID))
assert (effect_low <= effect_grid + 1e-12).all() and (
    effect_grid <= effect_high + 1e-12
).all()
assert (
    too_high + too_low < 0.10
), "too many curves lie outside their range: check the bootstrap"
assert (
    abs(effects_boot[:, :, plus10].mean() - effect_grid[:, plus10].mean()) < 0.1
), "the bootstrap is centred somewhere else"

assert (
    curves["delta_pts"].to_numpy() == np.tile(GRID[steps], len(base))
).all(), "the slider table must be in the order of the arrays"
curves["effect_low"] = effect_low[
    :, steps
].ravel()  # the range goes into the slider table
curves["effect_high"] = effect_high[:, steps].ravel()

print(f"widened to include the central curve: {too_high + too_low:.1%} of the values")
print(
    f"+10 points: central {effect_grid[:, plus10].mean():+.2f} °C | bootstrap mean {effects_boot[:, :, plus10].mean():+.2f} °C | mean range {effect_low[:, plus10].mean():+.2f} to {effect_high[:, plus10].mean():+.2f} °C"
)
width = effect_high[:, plus10] - effect_low[:, plus10]
print(
    f"width of the range at +10 points: median {np.median(width):.2f} °C, 90% of the districts between {np.percentile(width, 5):.2f} and {np.percentile(width, 95):.2f} °C"
)
print(
    f"districts whose whole range is below 0 at +10 points (sure to get cooler): {(effect_high[:, plus10] < 0).mean():.1%}"
)

## Step 3: The typical error of the absolute temperature
The range above is for the **change** (`effect`). The predicted surface temperature itself also has an error: the walk-forward exam of the modeling notebook predicted 2023, 2024 and 2025 with models that had not seen these years. The 10th and 90th percentile of the errors (measured minus predicted, real weather) tell what a prediction of the temperature of a district is typically off by. We also look at the **mean** error per year, because an error that is not centered on 0 is a bias.

In [ ]:
exam = pd.read_csv(MODELS / "exam_predictions.csv")
error = exam[TARGET] - exam["model_real_weather"]
error_low, error_high = np.percentile(error, [10, 90])
assert error_low < 0 < error_high

print(f"{len(exam)} exam predictions | RMSE {np.sqrt((error ** 2).mean()):.2f} °C")
print(
    f"error of a predicted temperature: 80% of the districts between {error_low:+.1f} and {error_high:+.1f} °C (10th and 90th percentile)"
)
print(
    "mean error (measured minus predicted) by year:",
    error.groupby(exam["year"]).mean().round(2).to_dict(),
)

In [ ]:
example = np.argsort(effect_grid[:, plus10])[
    [0, len(base) // 2, len(base) - 1]
]  # the strongest, a middle and the weakest effect at +10 points
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for i, color in zip(example, ["#2c6e91", "#F57C00", "#7b7b7b"]):
    label = f"{base.loc[i, 'district_name']} ({base.loc[i, 'city_name']})"
    axes[0].fill_between(GRID, effect_low[i], effect_high[i], color=color, alpha=0.2)
    axes[0].plot(GRID, effect_grid[i], color=color, lw=2, label=label)
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].set_xlabel("change of the green share (points)")
axes[0].set_ylabel("change of the surface temperature (°C)")
axes[0].set_title("Three districts with the range of the bootstrap (5-95%)")
axes[0].legend(fontsize=8)

axes[1].hist(width, bins=30, color="#F57C00", edgecolor="black", linewidth=0.5)
axes[1].set_xlabel("width of the range at +10 points (°C)")
axes[1].set_ylabel("districts")
axes[1].set_title("How wide are the ranges?")
plt.tight_layout()
plt.show()

**What we learned (Part 3)**
- **The size of the cooling is uncertain, the direction is not.** At +10 points the central estimate is -0.63 °C and the 5-95% range of a typical district is -0.89 to -0.37 °C; the range of a single district is typically 0.48 °C wide (90% of the districts between 0.31 and 0.88 °C). All 829 districts cool at +10 points in the model, and in 99.8% of them the whole range is below zero; the sign is built in by the monotone constraint, so what the bootstrap tells is *how much*.
- **The ranges are very different from district to district.** The figure shows Arapgir (Malatya), with a green share of 24%, at about -3 °C at +30 points and a wide range, and Adalar (İstanbul), which is already 82% green, that hardly changes. Districts with unusual features have steep and wide curves: read them as "a lot, but very uncertain". A single district curve is jagged: a tree model has seen only a few similar districts.
- **The bootstrap is centred a little below the central curve** (mean -0.61 °C against -0.63 °C: models that have seen only about two thirds of the cities react a little less strongly); only 2.9% of the values had to be widened to contain the central estimate.
- **The absolute temperature has a bias in the exam:** the models that had not seen 2023-2025 predicted too cool by 0.50, 0.22 and 0.21 °C on average (measured minus predicted; the model before the improvement of the modeling notebook: 0.90, 0.56 and 0.39 °C), and 80% of the errors lie between -1.2 and +1.8 °C. The later summers were hotter than the earlier years led the models to expect (a warming trend, a change in the satellite data, or something else: we cannot tell which). This matters when we speak about the *level* of a district; the **change** (`effect`) is the difference of two predictions of the same model and does not carry this bias.

# <span style="color:#F57C00">Part 4: Does the What-if Match What Really Happened?</span>

Everything so far is the model talking to itself. Here we compare it with **real changes**: between 2013-2015 and 2023-2025 the green share of many districts really changed, and we can see how their surface temperature changed.

## Step 1: The real changes
For every district: the mean of 2013-2015 and the mean of 2023-2025 (each with at least 2 reliable summers) of the green share and of the surface temperature. To remove the weather and the climate of the whole region, every change is taken **minus the mean change of the districts of the same city** (only cities with at least 3 such districts stay), so we only look at *which districts changed more than the others in their city*.

In [ ]:
periods = {"early": (2013, 2015), "late": (2023, 2025)}
means = pd.concat(
    {
        name: table[table["year"].between(*years)]
        .groupby("district_code")
        .agg(
            n_years=("year", "size"),
            green=("landsat_green_share", "mean"),
            lst=(TARGET, "mean"),
        )
        for name, years in periods.items()
    },
    axis=1,
).dropna()  # districts that are in both periods
means.columns = [
    f"{name}_{period}" for period, name in means.columns
]  # n_years_early, green_early, lst_early, ...
means = means[(means["n_years_early"] >= 2) & (means["n_years_late"] >= 2)]
means["city_code"] = table.groupby("district_code")["city_code"].first()

changes = means.assign(
    d_green=means["green_late"] - means["green_early"],
    d_lst=means["lst_late"] - means["lst_early"],
)
changes = changes[changes.groupby("city_code")["d_green"].transform("size") >= 3].copy()
for name in ("d_green", "d_lst"):
    changes[f"{name}_c"] = changes[name] - changes.groupby("city_code")[name].transform(
        "mean"
    )  # minus the city mean

assert len(changes) >= 800, "too few districts with a change"
print(f"{len(changes)} districts in {changes['city_code'].nunique()} cities")
print(
    f"change of the green share inside the city: standard deviation {changes['d_green_c'].std() * 100:.1f} points | {(changes['d_green_c'].abs() >= 0.05).sum()} districts changed by 5 points or more"
)
print(
    f"change of the surface temperature inside the city: standard deviation {changes['d_lst_c'].std():.2f} °C"
)

## Step 2: The observed slope
How much cooler did a district get, relative to its city, per +10 points of green share? The line is a least-squares fit through the districts. Districts of one city are neighbours and not independent, so the uncertainty comes from a **bootstrap over cities** (1000 draws).

We expect the observed slope to be **weaker than the model effect**: the green share is measured with noise (which weakens any slope: "regression dilution"), and the surface temperature of three summers is noisy.

In [ ]:
fit = stats.linregress(changes["d_green_c"], changes["d_lst_c"])
observed = fit.slope * 0.10  # °C per +10 points

rng = np.random.default_rng(SEED)
cities = changes["city_code"].unique()
by_city = changes.set_index("city_code")
boot_slopes = []
for _ in range(1000):
    sample = by_city.loc[
        rng.choice(cities, size=len(cities), replace=True)
    ]  # whole cities are drawn
    boot_slopes.append(np.polyfit(sample["d_green_c"], sample["d_lst_c"], 1)[0] * 0.10)
observed_low, observed_high = np.percentile(boot_slopes, [2.5, 97.5])

big = changes[
    changes["d_green_c"].abs() >= 0.05
]  # only the districts that changed clearly
big_slope = stats.linregress(big["d_green_c"], big["d_lst_c"]).slope * 0.10

assert fit.slope < 0 and fit.pvalue < 0.01, "more green should go with a cooler surface"
print(
    f"observed: {observed:+.2f} °C per +10 points (95% range {observed_low:+.2f} to {observed_high:+.2f}, correlation {fit.rvalue:+.2f})"
)
print(
    f"only the {len(big)} districts that changed by 5 points or more inside their city: {big_slope:+.2f} °C per +10 points"
)

## Step 3: District by district
Now the model. For each district we ask the model: *what would the late summers have been with the green share of the early years?* The difference to the real late prediction is the cooling that the model **attributes to the change of the green share**, with everything else at its late value. It is then taken minus the city mean in the same way. If the model is right, the observed change should be as large as the predicted one (slope 1 when the observed change is plotted against the predicted one).

The model for this check is trained on **2013-2022 only** (the final model has seen all years, so it has seen the late summers that we compare with; a model that has not seen them is the honest choice).

In [ ]:
seen = table["year"] <= 2022
check_model = new_final(FEATURES, card["settings"], seed=0, n_jobs=-1).fit(
    table.loc[seen, FEATURES], table.loc[seen, TARGET]
)

late_rows = table[table["year"].between(*periods["late"])].copy()
swapped = late_rows.copy()  # the late rows with the early green share
swapped["landsat_green_share"] = (
    swapped["district_code"]
    .map(changes["green_early"])
    .fillna(swapped["landsat_green_share"])
)
swapped = add_within_city(
    swapped, WITHIN_CITY_NEW
)  # whole city-years are in the table, so the within-city columns are right

late_rows["model_change"] = check_model.predict(
    late_rows[FEATURES]
) - check_model.predict(swapped[FEATURES])
changes["model_change"] = late_rows.groupby("district_code")["model_change"].mean()
changes["model_change_c"] = changes["model_change"] - changes.groupby("city_code")[
    "model_change"
].transform("mean")

model_slope = np.polyfit(changes["d_green_c"], changes["model_change_c"], 1)[0] * 0.10
calibration = stats.linregress(changes["model_change_c"], changes["d_lst_c"])
correlation = stats.pearsonr(changes["model_change_c"], changes["d_lst_c"])

assert (
    correlation.statistic > 0 and correlation.pvalue < 0.01
), "the district-by-district prediction must go in the right direction"
assert (
    0.2 < calibration.slope < 2.0
), "the observed change should be of the same order as the predicted one"
print(
    f"the model attributes {model_slope:+.2f} °C per +10 points to the changes of these districts (observed {observed:+.2f})"
)
print(
    f"observed change against predicted change: slope {calibration.slope:.2f} (1 = perfect), correlation {correlation.statistic:+.2f}, p = {correlation.pvalue:.1e}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = changes["d_green_c"] * 100
axes[0].scatter(x, changes["d_lst_c"], s=8, alpha=0.4, color="#7b7b7b")
line = np.array([x.min(), x.max()])
axes[0].plot(
    line,
    fit.intercept + fit.slope * line / 100,
    color="#F57C00",
    lw=2,
    label=f"observed: {observed:+.2f} °C per +10 points",
)
axes[0].plot(
    line,
    model_slope * line / 10,
    color="#2c6e91",
    lw=2,
    ls="--",
    label=f"model: {model_slope:+.2f} °C per +10 points",
)
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].set_xlabel("change of the green share inside the city (points)")
axes[0].set_ylabel("change of the surface temperature inside the city (°C)")
axes[0].set_ylim(-4, 4)  # a few districts lie outside this frame
axes[0].set_title("Every point is a district")
axes[0].legend()

groups = changes.groupby(
    pd.qcut(changes["d_green_c"], 10)
)  # ten groups of equal size, from the biggest loss to the biggest gain of green
tens = groups[["d_green_c", "d_lst_c", "model_change_c"]].mean()
print(
    (tens * [100, 1, 1])
    .round(2)
    .rename(
        columns={
            "d_green_c": "green change (points)",
            "d_lst_c": "observed (°C)",
            "model_change_c": "model (°C)",
        }
    )
    .to_string(index=False)
)
axes[1].plot(
    tens["d_green_c"] * 100, tens["d_lst_c"], "o-", color="#F57C00", label="observed"
)
axes[1].plot(
    tens["d_green_c"] * 100,
    tens["model_change_c"],
    "s--",
    color="#2c6e91",
    label="model",
)
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set_xlabel("mean change of the green share of the group (points)")
axes[1].set_ylabel("mean change of the surface temperature (°C)")
axes[1].set_title("Ten equal groups, from the biggest loss to the biggest gain")
axes[1].legend()
plt.tight_layout()
plt.show()

## Step 4: Robustness checks on other periods and with controls

The decade above is one comparison. Three cheap checks use the same data and train no model:

1. **Four periods, fixed in advance:** (2013-2015 → 2018-2020), (2013-2015 → 2023-2025), (2016-2018 → 2021-2023) and (2018-2020 → 2023-2025). Each estimate is the same city-centered association; the interval resamples whole cities. A slope that exists in only one of them would be fragile.
2. **Weather and coverage controls:** the same comparison, adjusted for the change in summer air temperature and hot days, in spring precipitation and in satellite coverage. These are *descriptive* sensitivity checks: a control changes the number; it does not make the number causal (control selection needs a causal rationale).
3. **A panel view:** every reliable district-summer, with **district** and **city-year fixed effects** — the comparison inside a district over time and inside a city in a year, again with a city bootstrap. Still an association: the timing of any planting is not in the data, so no prior-trend and no causal claim.
4. **Where does the association come from:** the decade comparison by region and by greenness at the start. Subgroups with fewer than 30 districts or 5 cities are withheld instead of reported.

Everything in this step is written to `data/models/observational_validation.json`; the first-period district changes go to `data/models/historical_changes.csv`; and the reference shown in the application (observed association next to the model number) is written into the model card.

In [ ]:
PERIOD_PAIRS = [
    ((2013, 2015), (2023, 2025)),  # the decade of Part 4 first: it is also the reference of the app
    ((2013, 2015), (2018, 2020)),
    ((2016, 2018), (2021, 2023)),
    ((2018, 2020), (2023, 2025)),
]


def changes_for(early, late, min_years=2):
    # per district: the mean green share and surface temperature of both periods, and the changes inside the city
    values = {"landsat_green_share": "green", TARGET: "lst"}
    values.update({name: name for name in WEATHER + ["landsat_cover"] if name in table})
    means = []
    for label, years in (("early", early), ("late", late)):
        rows = table[table["year"].between(*years)]
        grouped = rows.groupby("district_code")[list(values)].mean().rename(columns=values)
        grouped["n_years"] = rows.groupby("district_code")["year"].nunique()
        means.append(grouped.add_suffix(f"_{label}"))
    changes = means[0].join(means[1], how="inner")
    changes = changes[
        (changes["n_years_early"] >= min_years) & (changes["n_years_late"] >= min_years)
    ].dropna(subset=["green_early", "green_late", "lst_early", "lst_late"])
    changes["city_code"] = table.groupby("district_code")["city_code"].first().reindex(changes.index)
    changes = changes[changes.groupby("city_code")["city_code"].transform("size") >= 3].copy()
    changes["region"] = changes["city_code"].map(region_of_city)
    for name in values.values():
        changes[f"d_{name}"] = changes[f"{name}_late"] - changes[f"{name}_early"]
        changes[f"d_{name}_c"] = changes[f"d_{name}"] - changes.groupby("city_code")[f"d_{name}"].transform("mean")
    return changes


def green_coefficient(gram, rhs):
    # green coefficient per +10 points; None when the controls leave nothing to identify
    residual = gram[0, 0]
    if len(rhs) > 1:
        residual -= gram[0, 1:] @ np.linalg.pinv(gram[1:, 1:]) @ gram[1:, 0]
    if residual <= max(1e-14, abs(gram[0, 0]) * 1e-10):
        return None
    value = np.linalg.lstsq(gram, rhs, rcond=None)[0][0] * 0.10
    return float(value) if np.isfinite(value) else None


def city_association(x, y, cities, controls, rng, draws=1000):
    # fit once on the absorbed values, then resample whole city blocks and refit with their sums
    x = np.asarray(x, dtype=float).copy()
    if x.shape[1] > 1:
        scales = x[:, 1:].std(axis=0)
        x[:, 1:] /= np.where(scales > 0, scales, 1)
    unique = pd.unique(cities)
    grams, rhs = [], []
    for city in unique:
        selected = np.asarray(cities) == city
        grams.append(x[selected].T @ x[selected])
        rhs.append(x[selected].T @ y[selected])
    grams, rhs = np.asarray(grams), np.asarray(rhs)
    estimates = []
    for _ in range(draws):
        sampled = rng.integers(0, len(unique), size=len(unique))
        value = green_coefficient(grams[sampled].sum(axis=0), rhs[sampled].sum(axis=0))
        if value is not None:
            estimates.append(value)
    return dict(
        n_cities=int(len(unique)),
        controls=list(controls),
        association_per_10pts=green_coefficient(grams.sum(axis=0), rhs.sum(axis=0)),
        city_bootstrap_95=np.percentile(estimates, [2.5, 97.5]).tolist() if estimates else [None, None],
        bootstrap_valid_draws=len(estimates),
        bootstrap_draws=int(draws),
    )


def city_centered_association(changes, controls=(), rng=None):
    # the same comparison as Part 4: every change minus the mean change of its own city
    names = ["d_green", "d_lst"] + [f"d_{name}" for name in controls]
    sample = changes[["city_code"] + names].dropna()
    centered = sample[names] - sample.groupby("city_code")[names].transform("mean")
    x = centered[["d_green"] + [f"d_{name}" for name in controls]].to_numpy()
    result = city_association(x, centered["d_lst"].to_numpy(), sample["city_code"].to_numpy(), controls, rng)
    result["n_districts"] = int(len(sample))
    return result


In [ ]:
rng = np.random.default_rng(SEED)
period_results = {}
first_changes = None
for early, late in PERIOD_PAIRS:
    changes_p = changes_for(early, late)
    if first_changes is None:
        first_changes = changes_p  # the decade: it is also the table of the subgroups below
    estimates = {"city_centered": city_centered_association(changes_p, (), rng)}
    if all(name in table for name in WEATHER):
        estimates["climate_controls"] = city_centered_association(changes_p, WEATHER, rng)
        if "landsat_cover" in table:
            estimates["climate_and_coverage"] = city_centered_association(
                changes_p, WEATHER + ["landsat_cover"], rng
            )
    period_results[f"{early[0]}-{early[1]} to {late[0]}-{late[1]}"] = estimates

rows = []
for key, estimates in period_results.items():
    estimate = estimates["city_centered"]
    interval = estimate["city_bootstrap_95"]
    rows.append(
        {
            "period": key,
            "districts": estimate["n_districts"],
            "association (per +10 points)": estimate["association_per_10pts"],
            "95% over cities": f"{interval[0]:+.2f} to {interval[1]:+.2f}",
        }
    )
period_table = pd.DataFrame(rows).set_index("period")
print(
    "the decade with controls:",
    {
        name: (None if value["association_per_10pts"] is None else round(value["association_per_10pts"], 3))
        for name, value in period_results["2013-2015 to 2023-2025"].items()
    },
)
period_table

In [ ]:
def panel_association(draws=1000, seed=SEED, tolerance=1e-10, max_iterations=1000):
    # district and city-year fixed effects, absorbed by alternating demeaning (works for an
    # unbalanced panel), then the same city bootstrap on the absorbed values
    controls = [name for name in WEATHER + ["landsat_cover"] if name in table]
    columns = ["district_code", "city_code", "year", "landsat_green_share", TARGET] + controls
    sample = table[columns].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    assert not sample.duplicated(["district_code", "year"]).any()
    values = sample[["landsat_green_share", TARGET] + controls].astype(float)
    scales = np.maximum(values.std(ddof=0).to_numpy(), 1e-12)
    converged, iterations = False, 0
    for iterations in range(1, max_iterations + 1):
        previous = values.to_numpy().copy()
        values -= values.groupby(sample["district_code"]).transform("mean")
        values -= values.groupby([sample["city_code"], sample["year"]]).transform("mean")
        if np.max(np.abs(values.to_numpy() - previous) / scales) <= tolerance:
            converged = True
            break
    result = city_association(
        values[["landsat_green_share"] + controls].to_numpy(),
        values[TARGET].to_numpy(),
        sample["city_code"].to_numpy(),
        controls,
        np.random.default_rng(seed),
        draws,
    )
    result.update(
        n_observations=int(len(sample)),
        n_districts=int(sample["district_code"].nunique()),
        n_years=int(sample["year"].nunique()),
        demeaning_converged=converged,
        demeaning_iterations=int(iterations),
        fixed_effects=["district", "city_year"],
        bootstrap_unit="city",
        observational_only=True,
        interpretation="Descriptive conditional association; intervention timing and causal effects are not identified; no prior-trend claim.",
    )
    return result


panel = panel_association()
print(
    f"panel with district and city-year fixed effects: {panel['association_per_10pts']:+.3f} °C per +10 points "
    f"(city bootstrap {panel['city_bootstrap_95'][0]:+.3f} to {panel['city_bootstrap_95'][1]:+.3f}, "
    f"{panel['n_observations']} district-summers; demeaning converged: {panel['demeaning_converged']})"
)

In [ ]:
# the subgroups of the decade: by region and by greenness at the start
baseline = pd.qcut(first_changes["green_early"], 3, duplicates="drop")
stability = {}
for name, groups in (("region", first_changes["region"]), ("baseline_greenness", baseline)):
    records = []
    for group in groups.dropna().unique():
        selected = first_changes[groups == group]
        entry = dict(
            group=str(group),
            n_districts=int(len(selected)),
            n_cities=int(selected["city_code"].nunique()),
        )
        if len(selected) < 30 or selected["city_code"].nunique() < 5:
            entry["status"] = "insufficient subgroup size"
        else:
            entry["status"] = "reported"
            entry["estimate"] = city_centered_association(selected, (), rng)
        records.append(entry)
    stability[name] = records

diagnostics = dict(
    observational_only=True,
    bootstrap_unit="city",
    seed=int(SEED),
    min_observations_per_period=2,
    periods=[
        dict(
            early=list(early),
            late=list(late),
            estimates=period_results[f"{early[0]}-{early[1]} to {late[0]}-{late[1]}"],
        )
        for early, late in PERIOD_PAIRS
    ],
    stability=stability,
    panel_diagnostic=panel,
    interpretation="Associations and model sensitivities do not bound intervention effects.",
    controls_caveat=(
        "Contemporaneous weather/coverage controls are descriptive sensitivity checks; "
        "control selection needs a causal rationale."
    ),
)
(MODELS / "observational_validation.json").write_text(json.dumps(diagnostics, indent=2, default=float))
first_changes.to_csv(MODELS / "historical_changes.csv")

# the reference the application shows next to the model number: the decade, city-centered
reference = diagnostics["periods"][0]
observed_reference = reference["estimates"]["city_centered"]
card["observational_reference"] = dict(
    kind="observed_association",
    effect_per_10pts=observed_reference["association_per_10pts"],
    low=observed_reference["city_bootstrap_95"][0],
    high=observed_reference["city_bootstrap_95"][1],
    periods=dict(early=reference["early"], late=reference["late"]),
)
(MODELS / "model_card.json").write_text(json.dumps(card, indent=1, default=float))
print(
    f"reference written to the model card: {card['observational_reference']['effect_per_10pts']:+.3f} °C per +10 points "
    f"({card['observational_reference']['low']:+.3f} to {card['observational_reference']['high']:+.3f}), 2013-2015 to 2023-2025"
)
print(f"subgroups reported: {sum(entry['status'] == 'reported' for records in stability.values() for entry in records)} of {sum(len(records) for records in stability.values())}")

**What we learned (Part 4)**
- **The model and reality agree on the direction, but the model is steeper.** Between 2013-2015 and 2023-2025, districts whose green share rose more than the others in their city got cooler: **-0.48 °C per +10 points** (824 districts; 95% range from a bootstrap over cities -0.74 to -0.24 °C; -0.43 °C for the 164 districts that changed clearly, by 5 points or more). The model attributes **-0.72 °C per +10 points** to the same changes: about 1.5 times the observed value, at the steep end of the observed range; a fit of the observed on the predicted change of every district gives a slope of 0.61 (1 would be perfect). The check in the training years (modeling notebook, 2013-2015 against 2020-2022) showed the same for the model before the improvement of Part 5: observed -0.60 °C, XGBoost -0.78 °C; the final model is a little closer to reality than the model before Part 5 (-0.80 and slope 0.54 in the earlier version of this check).
- **How to read this (no causal bracket):** a part of the gap is expected — the green share is measured with noise (which weakens any observed slope, "regression dilution") and three summers of surface temperature are noisy — but we cannot separate that from a real overestimate by the model. Neither number is a calibration factor for the other: the observed association describes what happened between the districts that changed; the model number is what the what-if of Part 2 produces. The application shows the two side by side and does not scale one to the other.
- **The ten groups of districts (figure):** the districts that gained the most green (+6.9 points on average) got cooler by 0.43 °C, the model says 0.51 °C; the districts that lost the most green (-7.4 points) got hotter by 0.26 °C, the model says 0.48 °C. The model follows the direction and exaggerates at both ends, most where green was lost; in the middle, where the changes are small, the observed values are noisy.
- **For a single district the change is hard to predict:** the correlation between predicted and observed change is only +0.23 (it explains about 5% of the changes inside the cities), because most of the change of a district comes from other things (buildings, the weather of three summers, noise). The model is good for the *average* relation, not for the exact result in one district.
- The check is also an *association*: districts that got greener may have changed in other ways too.
- **Robustness (Step 4 below):** the negative association appears in every one of the four prespecified period comparisons and survives the weather and coverage controls; the panel view with district and city-year fixed effects gives a steeper, still negative coefficient. Numbers in the tables of Step 4.

# <span style="color:#F57C00">Part 5: How Much Green Does a District Need? The Heat-Target Green Need</span>

Planners ask the question the other way round: *we want the surface to be 1 °C cooler, how much green is that?* We answer it with the curves of Part 2: for every district we look for the **first step of the curve that reaches the target**.

**Two targets**
- **-1 °C:** the same cooling for every district.
- **To the city median:** the district is hotter than the median district of its city (its mean surface temperature of 2023-2025, from the table of Part 1); the target is to close that gap. Districts that are not hotter than the median have no such target (empty, not zero).

**Points and hectares.** The answer is in points of green share. A point of green share of a built-up area of 1 km² is 1 ha (1 km² = 100 ha, 1 point = 1/100), so **hectares = points × `urban_km2`** (the built-up area of the district, which is the area the surface temperature describes).

**When the target cannot be reached.** The curve stops at +30 points or at the highest share in the data. A district whose curve does not reach the target before that is **not reachable** (`reachable = False`); we never extend the curve beyond what the model has seen.

**The range** comes from the bootstrap curves of Part 3: `_low` uses the curve with the strongest cooling (the optimistic end: fewer points), `_high` the curve with the weakest cooling (more points).

## Step 1: Finding the first sufficient step
`points_needed` works on all districts at once. It takes the effect curves (one row per district, 0 to +30 points, on the grid of the **applied** change) and the target of every district, and returns the **first step whose cooling actually reaches the target**. A tree model is a staircase: an interpolated fraction between two steps is not a value the model has produced, so we do not report one — the answer is the first sufficient step at the 1-point resolution of the curve. Three small tests with curves where we know the answer come first.

In [ ]:
def points_needed(effect, target, points=None):
    # effect: one row per district, the effect of the APPLIED change of every step (non-increasing); target: the wanted cooling in °C, one value per district
    # points: the applied change of every step (0 ... +30 by default); returns the FIRST SUFFICIENT step and nan where the target is never reached
    cooling = -effect
    reached = cooling >= target[:, None]
    first = reached.argmax(axis=1)  # the first step that reaches the target
    rows = np.arange(len(cooling))
    step = np.arange(effect.shape[1], dtype=float) if points is None else np.asarray(points, dtype=float)
    grid = np.broadcast_to(step, effect.shape)
    return np.where(reached.any(axis=1), grid[rows, first], np.nan)


straight = -0.06 * np.arange(31)  # -0.6 °C per 10 points: 1 °C is first reached at step 17 (16 steps are only 0.96 °C)
weak = np.maximum(-0.06 * np.arange(31), -0.5)  # stops at -0.5 °C: 1 °C is never reached
jump = np.where(np.arange(31) < 10, 0.0, -1.2)  # nothing until 9 steps, then a jump to -1.2 °C at step 10
found = points_needed(np.vstack([straight, weak, jump]), np.ones(3))
assert found[0] == 17 and np.isnan(found[1]) and found[2] == 10
# where the physical cap limits the share, the answer is the applied amount of the first sufficient step
capped = points_needed(
    np.array([[0.0, -0.4, -1.2]]), np.ones(1), points=np.array([[0.0, 1.0, 1.6]])
)
assert capped[0] == 1.6  # the target is reached at the third step, but that step applies only 1.6 points
print("tests passed:", found, "| with a capped applied grid:", capped)

## Step 2: The targets and the scorecard
The scorecard has one row per district. The columns of this notebook are the state of the district, the what-if at +10 points with its range, and the two heat targets. (The risk notebook adds the risk columns and saves the complete file.)

In [ ]:
zero_step = list(GRID).index(0)
effect_plus, low_plus, high_plus = (
    curve[:, zero_step:] for curve in (effect_grid, effect_low, effect_high)
)  # 0 ... +30 points

base["lst_recent"] = base["district_code"].map(
    table[table["year"].between(2023, 2025)].groupby("district_code")[TARGET].mean()
)
gap = (
    base["lst_recent"] - base.groupby("city_code")["lst_recent"].transform("median")
).to_numpy()  # hotter than the median district of the city by ...

scorecard = base[
    ["district_code", "city_code", "district_name", "city_name", "region"]
].copy()
scorecard["latest_year"] = base["year"]
scorecard["urban_km2"] = base["urban_km2"]
scorecard["green_share"] = base["landsat_green_share"]
scorecard["lst_recent"] = base["lst_recent"]
(
    scorecard["effect_10pts"],
    scorecard["effect_10pts_low"],
    scorecard["effect_10pts_high"],
) = (curve[:, plus10] for curve in (effect_grid, effect_low, effect_high))

applied_plus = (
    applied_grid[:, zero_step:] * 100
)  # the change really applied at every step, in POINTS (the unit the solver returns)
for suffix, curve in (("", effect_plus), ("_low", low_plus), ("_high", high_plus)):
    points = points_needed(curve, np.ones(len(base)), points=applied_plus)  # target: 1 °C cooler
    scorecard[f"green_needed_1c_pts{suffix}"] = points
    scorecard[f"hectares_1c{suffix}"] = points * scorecard["urban_km2"]
scorecard["reachable_1c"] = scorecard["green_needed_1c_pts"].notna()

points = points_needed(
    effect_plus, np.where(gap > 0, gap, np.nan), points=applied_plus
)  # target: down to the city median (nan where the district is not hotter than the median)
scorecard["green_needed_median_pts"] = points
scorecard["hectares_median"] = points * scorecard["urban_km2"]
scorecard["reachable_median"] = pd.Series(~np.isnan(points), dtype="boolean").where(
    gap > 0
)  # empty where there is no target
scorecard["small_area"] = base["urban_km2"] < 1  # a warning flag: the measured built-up area is small

print(
    f"{len(scorecard)} districts | -1 °C reachable: {scorecard['reachable_1c'].mean():.1%} | hotter than the city median: {(gap > 0).sum()} districts, of which reachable: {scorecard['reachable_median'].mean():.1%}"
)

### Scenario support: is there comparable data?

Physical clipping (0 to `MAX_GREEN`, and +30 points) keeps a scenario possible, but not necessarily **supported**: the combination of green share, climate, elevation and urban form may simply not exist in the data the model learned from. The check compares a scenario with the nearest comparable rows:

- the scenario must stay inside the **global range** of the green share and of the within-city green share;
- it must stay inside the **regional range** of the green share;
- the **joint distance** to the nearest row of another district in the same region (green share, within-city green share, climate normals, elevation, built share, built-up size) must not exceed a radius calibrated by leaving every district's own rows out (95th percentile).

A flag means "compare with care": the answer is then an **exploratory extrapolation**, not an observed-data-supported estimate. The saved +10 and +30 scenarios are checked here; the application runs the same check for the scenario the user asks for.

In [ ]:
SUPPORT_CONTEXT = ["temp_clim", "precip_clim", "elevation_m", "built_share_2020", "log_urban_km2"]
SUPPORT_GREEN = ["landsat_green_share", "landsat_green_share_vs_city"]


class ScenarioSupport:
    """Regional joint distances, calibrated by leaving every district's own rows out.
    Passing the checks means no weak support was detected in these columns; it guarantees neither
    feasibility nor causality."""

    def __init__(self, table):
        self.columns = SUPPORT_GREEN + [name for name in SUPPORT_CONTEXT if name in table]
        self.bounds = table[SUPPORT_GREEN].agg(["min", "max"])
        values = table[self.columns].to_numpy(dtype=float)
        self.mean, self.scale = values.mean(axis=0), values.std(axis=0)
        self.scale[self.scale == 0] = 1
        scaled = (values - self.mean) / self.scale
        regions = table["city_code"].map(region_of_city).fillna("unknown").to_numpy()
        self.references = {}
        for region in np.unique(regions):
            indexes = np.flatnonzero(regions == region)
            codes = table.iloc[indexes]["district_code"].to_numpy()
            reference = scaled[indexes]
            neighbor = NearestNeighbors(n_jobs=1).fit(reference)
            k = min(
                len(indexes), int(table.iloc[indexes].groupby("district_code").size().max()) + 1
            )
            distances, neighbors = neighbor.kneighbors(reference, n_neighbors=k)
            different = codes[neighbors] != codes[:, None]
            nearest_peer = np.min(np.where(different, distances, np.inf), axis=1)
            finite = nearest_peer[np.isfinite(nearest_peer)]
            radius = max(float(np.percentile(finite, 95)), 1e-8) if len(finite) else None
            green = table.iloc[indexes]["landsat_green_share"]
            self.references[region] = (neighbor, radius, float(green.min()), float(green.max()))

    def check(self, rows, new_share, relative_green):
        changed = rows[self.columns].copy()
        changed[SUPPORT_GREEN[0]], changed[SUPPORT_GREEN[1]] = new_share, relative_green
        values = (changed.to_numpy(dtype=float) - self.mean) / self.scale
        regions = rows["city_code"].map(region_of_city).fillna("unknown").to_numpy()
        results = []
        for i, region in enumerate(regions):
            reasons = []
            if not self.bounds.loc["min", SUPPORT_GREEN[0]] - 1e-10 <= new_share[i] <= self.bounds.loc["max", SUPPORT_GREEN[0]] + 1e-10:
                reasons.append("green_outside_global_range")
            if not self.bounds.loc["min", SUPPORT_GREEN[1]] - 1e-10 <= relative_green[i] <= self.bounds.loc["max", SUPPORT_GREEN[1]] + 1e-10:
                reasons.append("relative_green_outside_global_range")
            distance = radius = None
            if region in self.references:
                neighbor, radius, low, high = self.references[region]
                distance = float(neighbor.kneighbors(values[[i]], n_neighbors=1)[0][0, 0])
                if not low - 1e-10 <= new_share[i] <= high + 1e-10:
                    reasons.append("green_outside_region_range")
                if radius is None:
                    reasons.append("insufficient_reference_districts")
                elif distance > radius + 1e-10:
                    reasons.append("weak_joint_support")
            else:
                reasons.append("region_missing")
            results.append(dict(supported=not reasons, reasons=reasons, nearest_distance=distance, reference_radius=radius))
        return results


support = ScenarioSupport(table)
checks_by_step = {}
for step in (10, 30):
    index = list(GRID).index(step)
    relative = base["landsat_green_share_vs_city"].to_numpy() + applied_grid[:, index] * (
        1 - 1 / base["n_districts"].to_numpy()
    )
    checks_by_step[step] = support.check(base, new_share[:, index], relative)
    scorecard[f"support_{step}pts"] = [check["supported"] for check in checks_by_step[step]]

print(
    f"pass the support checks: +10 points {scorecard['support_10pts'].mean():.1%} of the districts, "
    f"+30 points {scorecard['support_30pts'].mean():.1%}"
)
reasons = pd.Series([reason for check in checks_by_step[30] for reason in check["reasons"]]).value_counts()
print("reasons at +30 points:", reasons.to_dict())


## Step 3: Tests
1. The first sufficient step is checked **with the engine itself** for 30 random districts: the returned amount reaches the target, and no smaller applied amount does.
2. The low value is never above the central one and the central never above the high one; where the central value is reachable, the low value is too; where the high value is reachable, the central value is too.
3. Hectares are points × built-up area, and no value is missing except where a target does not exist or is not reachable.

In [ ]:
reachable = np.flatnonzero(scorecard["reachable_1c"])
rng = np.random.default_rng(1)
for i in rng.choice(reachable, size=30, replace=False):
    needed = scorecard.loc[i, "green_needed_1c_pts"]
    one = base.loc[[i]]
    assert (
        what_if(one, [needed / 100])[2][0, 0] <= -1 + 1e-9
    ), "the returned amount does not reach the target"
    assert (
        effect_plus[i][applied_plus[i] < needed - 1e-9].max() > -1 + 1e-9
    ), "a smaller applied amount already reached the target"

low, mid, high = (scorecard[f"green_needed_1c_pts{s}"] for s in ("_low", "", "_high"))
assert ((low <= mid + 1e-9) | mid.isna()).all() and (
    (mid <= high + 1e-9) | high.isna()
).all()
assert (low.notna() | mid.isna()).all() and (mid.notna() | high.isna()).all()
assert np.allclose(
    scorecard["hectares_1c"].dropna(), (mid * scorecard["urban_km2"]).dropna()
)
assert scorecard["reachable_median"].isna().sum() == (gap <= 0).sum()
not_marked = scorecard.drop(
    columns=[
        c
        for c in scorecard
        if c.startswith(("green_needed", "hectares", "reachable_median"))
    ]
)
assert (
    not_marked.isna().sum().sum() == 0
), "a column that must be complete has missing values"

print("30 districts checked with the engine itself: the returned amount reaches the target and no smaller applied amount does")
print(
    scorecard.loc[
        reachable[:5],
        [
            "district_name",
            "city_name",
            "urban_km2",
            "green_needed_1c_pts",
            "hectares_1c",
        ],
    ]
    .round(1)
    .to_string(index=False)
)

## Step 4: What do the targets look like?

In [ ]:
mid = scorecard["green_needed_1c_pts"]
print(
    f"-1 °C: {mid.notna().sum()} of {len(mid)} districts can reach it within +30 points ({mid.notna().mean():.1%}); with the weak end of the range {scorecard['green_needed_1c_pts_high'].notna().mean():.1%}, with the strong end {scorecard['green_needed_1c_pts_low'].notna().mean():.1%}"
)
print(
    f"points of green share needed (reachable districts): median {mid.median():.1f}, middle half {mid.quantile(0.25):.1f} to {mid.quantile(0.75):.1f}, 90th percentile {mid.quantile(0.9):.1f}"
)
hectares = scorecard["hectares_1c"]
print(
    f"hectares: median {hectares.median():.0f}, middle half {hectares.quantile(0.25):.0f} to {hectares.quantile(0.75):.0f}, 90th percentile {hectares.quantile(0.9):.0f} (the EDA estimate with one straight line for all districts: about 17 points and 66 ha for the median district)"
)
print(
    f"to the city median: median {scorecard['green_needed_median_pts'].median():.1f} points and {scorecard['hectares_median'].median():.0f} ha, for the districts that are hotter than the median"
)
unreached = (
    scorecard.groupby("reachable_1c")[["green_share", "effect_10pts", "urban_km2"]]
    .median()
    .round(2)
)
print(
    "median of the districts that do not (False) and do (True) reach -1 °C:\n",
    unreached.to_string(),
)
print(
    f"districts whose model effect is as strong as the -0.8 of the literature value: {(scorecard['effect_10pts'] <= -0.8).mean():.1%}"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(mid.dropna(), bins=30, color="#F57C00", edgecolor="black", linewidth=0.5)
axes[0].axvline(
    mid.median(), color="black", ls="--", label=f"median {mid.median():.0f} points"
)
axes[0].set_xlabel("points of green share needed for -1 °C")
axes[0].set_ylabel("districts")
axes[0].set_title(f"{mid.isna().sum()} districts do not reach -1 °C within +30 points")
axes[0].legend()

axes[1].hist(
    scorecard["effect_10pts"],
    bins=30,
    color="#F57C00",
    edgecolor="black",
    linewidth=0.5,
)
axes[1].axvline(
    scorecard["effect_10pts"].mean(),
    color="black",
    lw=2,
    label=f"model: {scorecard['effect_10pts'].mean():+.2f}",
)
axes[1].axvline(
    observed,
    color="#2c6e91",
    lw=2,
    ls="--",
    label=f"observed 2013-2015 to 2023-2025: {observed:+.2f}",
)
axes[1].axvline(
    -0.8,
    color="#7b7b7b",
    lw=2,
    ls=":",
    label="literature value of the project plan: -0.80",
)
axes[1].set_xlabel("change of the surface temperature for +10 points (°C)")
axes[1].set_ylabel("districts")
axes[1].set_title("The model against the observed and the literature value")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 5: Save
Two files in `data/results/` (small, under 1 MB): the slider table `what_if_curves.csv` and the first part of `district_scorecard.csv`, which the risk notebook completes (its columns are explained there).

`what_if_curves.csv` has one row per district and step: `district_code`, `district_name`, `city_name`, `delta_pts` (change of the green share in points, -20 to +30 in steps of 5), `new_green_share`, `predicted_lst` (predicted surface temperature in °C), `effect` (change against today in °C) and `effect_low` / `effect_high` (the 5-95% range of the effect).

In [ ]:
RESULTS.mkdir(exist_ok=True)
slider = curves[
    [
        "district_code",
        "district_name",
        "city_name",
        "delta_pts",
        "new_green_share",
        "predicted_lst",
        "effect",
        "effect_low",
        "effect_high",
    ]
]
assert (
    len(slider) == 829 * 11
    and slider.notna().all().all()
    and slider["delta_pts"].nunique() == 11
)
slider.round(4).to_csv(RESULTS / "what_if_curves.csv", index=False)
scorecard.to_csv(
    RESULTS / "district_scorecard.csv", index=False
)  # full precision: the risk notebook recomputes its ranks from this file; rounding can flip near-tied ranks
for name in ("what_if_curves.csv", "district_scorecard.csv"):
    print(f"{name}: {(RESULTS / name).stat().st_size / 1e3:.0f} KB")
print(f"scorecard so far: {scorecard.shape[0]} rows, {scorecard.shape[1]} columns")

**What we learned (Part 5)**
- **-1 °C is reachable for 785 of the 829 districts (94.7%) within +30 points of green share.** The uncertainty reaches the question itself: with the weak end of the range 77.7% can reach it, with the strong end 98.7%.
- **A typical district needs about 17 points more green share (middle half 14.0 to 21.0) for -1 °C, which is 66 ha for the median district** (middle half 27 to 188 ha, 90th percentile 390 ha). The hectares grow with the built-up area, so the largest figures belong to big-city districts (Çankaya in Ankara about 1,550 ha, Aksu in Antalya about 1,350 ha, Akdeniz in Mersin about 1,292 ha). This is about the **rough estimate of the EDA** with one straight line (17 points, 66 ha), because the effect of the model (-0.63 °C per +10 points) is about the slope of that line (-0.6). With the observed association of Part 4 (-0.48 °C) a district would need about 21 points; that is a different, observational calculation and **not a bracket around the model number** (Part 4 explains why).
- **The 44 districts that do not reach -1 °C respond less:** their median effect is -0.30 °C per +10 points (against -0.62 °C) and their median green share is 42% (against 25%). 26 of the 44 are in Marmara, where the effect of the model is weakest (-0.47 °C), and none of the 9 districts with more than 80% green can reach it.
- **To the city median:** 398 districts are hotter than the median district of their city; to close the gap the median of them needs 12.0 points (54 ha), and 83.2% can reach it.
- **The literature value:** the project plan assumed about -0.8 °C per +10 points. The model gives -0.63 °C (22.6% of the districts reach -0.8 °C or more) and the real changes -0.48 °C. With -0.8 a district would need 12.5 points for -1 °C; with the model 17.0 (median); with the observed association about 21.
- **What these numbers are:** the change of the *surface temperature of the built-up area* in a typical recent summer, not of the air temperature; an association from the comparison of districts, not an experiment; one district at a time (for a whole city that gets greener together the cooling per point is about 30% smaller, see Part 2). They are for ranking and for the order of magnitude, with the range of Part 3 next to them.